# தூண்டுதல் பொறியியல் அறிமுகம்
தூண்டுதல் பொறியியல் என்பது இயற்கை மொழி செயலாக்க பணிகளுக்கான தூண்டுதல்களை வடிவமைத்து மேம்படுத்தும் செயல்முறை. இது சரியான தூண்டுதல்களை தேர்ந்தெடுப்பது, அவர்களின் அளவுருக்களை சரிசெய்வது மற்றும் அவற்றின் செயல்திறனை மதிப்பீடு செய்வதை உள்ளடக்கியது. NLP மாதிரிகளில் உயர்ந்த துல்லியம் மற்றும் திறமையை அடைவதற்கு தூண்டுதல் பொறியியல் அவசியமானது. இந்த பகுதியில 우리는 OpenAI மாதிரிகள் பயன்படுத்தி தூண்டுதல் பொறியியலின் அடிப்படைகளை ஆராயப்போகிறோம்.


### பயிற்சி 1: குறியீட்டாக்கம்
OpenAI-இன் ஒரு திறந்த மூல விரைவு குறியீட்டாக்கி tiktoken பயன்படுத்தி குறியீட்டாக்கத்தை ஆராய்க
இன்னும் உதாரணங்களுக்காக [OpenAI Cookbook](https://github.com/openai/openai-cookbook/blob/main/examples/How_to_count_tokens_with_tiktoken.ipynb?WT.mc_id=academic-105485-koreyst) பார்க்கவும்.


In [ ]:
# EXERCISE:
# 1. Run the exercise as is first
# 2. Change the text to any prompt input you want to use & re-run to see tokens

import tiktoken

# Define the prompt you want tokenized
text = f"""
Jupiter is the fifth planet from the Sun and the \
largest in the Solar System. It is a gas giant with \
a mass one-thousandth that of the Sun, but two-and-a-half \
times that of all the other planets in the Solar System combined. \
Jupiter is one of the brightest objects visible to the naked eye \
in the night sky, and has been known to ancient civilizations since \
before recorded history. It is named after the Roman god Jupiter.[19] \
When viewed from Earth, Jupiter can be bright enough for its reflected \
light to cast visible shadows,[20] and is on average the third-brightest \
natural object in the night sky after the Moon and Venus.
"""

# Set the model you want encoding for
encoding = tiktoken.encoding_for_model("gpt-4o")

# Encode the text - gives you the tokens in integer form
tokens = encoding.encode(text)
print(tokens);

# Decode the integers to see what the text versions look like
[encoding.decode_single_token_bytes(token) for token in tokens]


### பயிற்சி 2: Microsoft Foundry Models முக்கிய அமைவுகளை சரிபார்க்கவும்

> **குறிப்பு:** GitHub Models 2026 ஜூலை மாதத்தின் இறுதியில் நிறுத்தப்பட உள்ளது. இந்த பயிற்சி அதற்கு பதிலாக [Microsoft Foundry Models](https://ai.azure.com/catalog/models?WT.mc_id=academic-105485-koreyst)ஐ பயன்படுத்துகிறது, இது அதேவாறான இலவசமாக முயற்சி செய்யும் மாதிரி கலவையையும் Azure AI Inference SDK அனுபவத்தையும் வழங்குகிறது.

கீழே உள்ள கோ드를 இயக்கி உங்கள் Microsoft Foundry Models முனைப்பு சரியாக அமைக்கப்பட்டுள்ளதா என விசாரணை செய்யவும். இந்தக் கோடு ஒரு எளிய அடிப்படைக் கேள்வியை முயற்சி செய்து முடிவைச் சரிபார்க்கிறது. `oh say can you see` என உள்ளீடு செய்யப்படும்போது `by the dawn's early light..` போன்ற வரிகளுடன் முடிவடைய வேண்டும்.


In [ ]:
import os
from azure.ai.inference import ChatCompletionsClient
from azure.ai.inference.models import SystemMessage, UserMessage
from azure.core.credentials import AzureKeyCredential

# Get these from your Microsoft Foundry project's "Overview" page
token = os.environ["AZURE_INFERENCE_CREDENTIAL"]
endpoint = os.environ["AZURE_INFERENCE_ENDPOINT"]

# temperature/top_p need a non-reasoning model (gpt-5 rejects them), so use a Llama model here.
model_name = os.environ.get("AZURE_INFERENCE_CHAT_MODEL", "Llama-3.3-70B-Instruct")

client = ChatCompletionsClient(
    endpoint=endpoint,
    credential=AzureKeyCredential(token),
)

def get_completion(prompt, client, model_name, temperature=1.0, max_tokens=1000, top_p=1.0):
    response = client.complete(
        messages=[
            {
                "role": "system",
                "content": "You are a helpful assistant.",
            },
            {
                "role": "user",
                "content": prompt,
            },
        ],
        model=model_name,
        temperature=temperature,
        max_tokens=max_tokens,
        top_p=top_p
    )
    return response.choices[0].message.content

## ---------- Call the helper method

### 1. Set primary content or prompt text
text = f"""
oh say can you see
"""

### 2. Use that in the prompt template below
prompt = f"""
```{text}```
"""

## 3. Run the prompt
response = get_completion(prompt, client, model_name)
print(response)


That line is the opening lyric of "The Star-Spangled Banner," the national anthem of the United States, written by Francis Scott Key. If you'd like more information or analysis, feel free to ask!


### பயிற்சி 3: ஊடுருவல்கள்
ஒரு தலைப்பை பற்றி அசல் இல்லை அல்லது அதன் முன்னர் பயிற்சி பெற்ற தரவுத்தொகுப்பை அப்புறப்படுத்தியது (சமீபத்தியவை) காரணமாக தெரிந்திராத தலைப்புகள் குறித்து ஒரு குறிப்பு தொடர்பாக LLM-க்கு முடிவுகளை திருப்பும்போது என்ன நடக்கும் என்பதை ஆராய்ந்து பார்க்கவும். வேறு குறிப்பு அல்லது வேறு மாதிரியை முயற்சித்தால் பதில் எப்படி மாறும் என்பதை பார்க்கவும்.


In [ ]:

## Set the text for simple prompt or primary content
## Prompt shows a template format with text in it - add cues, commands etc if needed
## Run the completion 
text = f"""
generate a lesson plan on the Martian War of 2076.
"""

prompt = f"""
```{text}```
"""

response = get_completion(prompt, client, model_name)
print(response)

### பயிற்சி 4: hướng dẫn அடிப்படையிலான  
முதன்மையான உள்ளடக்கத்தை அமைக்க "text" மாறியைப் பயன்படுத்தவும்  
மற்றும் அந்த முதன்மையான உள்ளடக்கத்துடன் தொடர்புடைய ஒரு hướng dẫn வழங்க "prompt" மாறியைப் பயன்படுத்தவும்.  

இங்கே நாம் மாடலை இரண்டாம் வகுப்பு மாணவருக்காக உரையை சுருக்குமாறு கேட்கிறோம்  


In [4]:
# Test Example
# https://platform.openai.com/playground/p/default-summarize

## Example text
text = f"""
Jupiter is the fifth planet from the Sun and the \
largest in the Solar System. It is a gas giant with \
a mass one-thousandth that of the Sun, but two-and-a-half \
times that of all the other planets in the Solar System combined. \
Jupiter is one of the brightest objects visible to the naked eye \
in the night sky, and has been known to ancient civilizations since \
before recorded history. It is named after the Roman god Jupiter.[19] \
When viewed from Earth, Jupiter can be bright enough for its reflected \
light to cast visible shadows,[20] and is on average the third-brightest \
natural object in the night sky after the Moon and Venus.
"""

## Set the prompt
prompt = f"""
Summarize content you are provided with for a second-grade student.
```{text}```
"""

## Run the prompt
response = get_completion(prompt, client, model_name)
print(response)

Jupiter is the fifth planet from the Sun and the biggest one in our Solar System. It's made of gas and is much bigger than all the other planets put together! You can see Jupiter in the night sky because it's very bright. People have noticed it for a really long time and named it after a Roman god.


### பயிற்சி 5: சிக்கலான அறிவுரை
கணினி, பயனர் மற்றும் உதவியாளர் செய்திகள் உள்ள கோரிக்கையை முயற்சிக்கவும்
கணினி உதவியாளர் சூழலை அமைக்கிறது
பயனர் & உதவியாளர் செய்திகள் பலமுறை உரையாடல் சூழலை வழங்குகின்றன

உதவியாளர் தனித்துவம் கணினி சூழலில் "வெறுக்கத்தகுந்த" என அமைக்கப்பட்டிருப்பதை கவனிக்கவும்.
வேறு தனித்துவ சூழலை முயற்சிக்கவும். அல்லது உள்ளீடு/வெளியீடு செய்திகளின் வேறு தொடர் முயற்சிக்கவும்


In [5]:
response = client.complete(
    model=model_name,
    messages=[
        {"role": "system", "content": "You are a sarcastic assistant."},
        {"role": "user", "content": "Who won the world series in 2020?"},
        {"role": "assistant", "content": "Who do you think won? The Los Angeles Dodgers of course."},
        {"role": "user", "content": "Where was it played?"}
    ]
)
print(response.choices[0].message.content)

Oh, you mean the famous 2020 World Series that wasn’t in a regular location? That was the year they played in the glamorous Arlington, Texas, at Globe Life Field.


### பயிற்சி: உங்கள் உள்ளுணர்வை ஆராயுங்கள்
மேலே உள்ள எடுத்துக்காட்டுகள் புதிய ஊக்கங்களை (எளிமையானவை, சிக்கலானவை, அறிவுரைகள் போன்றவை) உருவாக்க நீங்கள் பயன்படுத்தக்கூடிய வடிவங்களை அளிக்கின்றன - எடுத்துக்காட்டுகள், குறிப்புகள் மற்றும் மேலும் பல பேசியுள்ள மற்ற யுக்திகளை ஆராய மற்ற பயிற்சிகளை உருவாக்க முயற்சிக்கவும்.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**மறுப்பு**:
இந்த ஆவணம் AI மொழிபெயர்ப்பு சேவை [Co-op Translator](https://github.com/Azure/co-op-translator) பயன்படுத்தி மொழிபெயர்க்கப்பட்டுள்ளது. நாங்கள் துல்லியத்திற்காக முயற்சி செய்துள்ளோம், ஆனால் தானாக செய்யப்படும் மொழிபெயர்ப்புகளில் பிழைகள் அல்லது தவறுகள் இருக்கலாம் என்பதை கவனத்தில் கொள்ளவும். அசல் ஆவணம் அதன் தாய்மொழியில் அதிகாரப்பூர்வ ஆதாரமாக கருதப்பட வேண்டும். முக்கியமான தகவல்களுக்கு, தொழில்நுட்பமான மனித மொழிபெயர்ப்பு பரிந்துரைக்கப்படுகிறது. இந்த மொழிபெயர்ப்பைப் பயன்படுத்துவதால் ஏற்படும் எந்த தவறான புரிதல்கள் அல்லது தவறான விளக்கத்திற்கும் நாங்கள் பொறுப்பில்வில்லை.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
